# TinyLlama Oracle LoRA — **V12**
## ASKSMART · Fine-tuning NLP → SQL Oracle

> **Version V12** — Benchmark V11 = 5/10 (50%). Objectif V12 : **90%+**
>
> Failles corrigées : α `WHERE LOGON` sans `ACTION_NAME=` · β heures→jours · δ double filtre acteur+objet · ε USERHOST
>
> Nouveautés : dataset enrichi ~11 500 ex · ré-équilibrage ciblé · SYSTEM_PROMPT renforcé · post_process_sql amélioré

---

| Cellule | Description |
|---|---|
| **1** | Installation des dépendances (pip + redémarrage auto) |
| **2** | Vérification GPU + versions |
| **3** | Téléchargement TinyLlama-1.1B-Chat-v1.0 |
| **4** | Génération table `oracle_audit_trail.csv` (500 lignes) |
| **5** | Dataset base V9 (9 000 exemples) |
| **6** | Patch V10 — Blocs C-F (mois nommés, OBJECT_NAME, actions, langage métier) |
| **7** | Patch V11 — Blocs G-H (temporel paramétrique, error-driven) |
| **8** | **NOUVEAU V12** — Blocs I-L (failles α β δ ε + enrichissement global) |
| **9** | Fusion + export dataset V12 (~9 000 ex rééquilibrés) |
| **10** | Fine-tuning LoRA V12 (r=32, alpha=64, 4 époques) |
| **11** | Rebuild ZIP LoRA |
| **12** | Chargement LoRA V12 + fonctions d'inférence |
| **13** | Aperçu table audit |
| **14** | **Benchmark V12** — 10 questions terrain |
| **15** | Rapport de performance global |


In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Installation des dépendances                    ║
# ║  Exécuter SEULE puis attendre le redémarrage automatique     ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys

subprocess.run([sys.executable,"-m","pip","uninstall","-y","numpy"], capture_output=True)
subprocess.run([sys.executable,"-m","pip","cache","purge"], capture_output=True)

subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir",
               "numpy>=2.0.0,<2.1.0"], check=True)

subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir",
               "transformers>=4.40.0",
               "peft>=0.10.0",
               "accelerate>=0.28.0",
               "datasets>=2.18.0",
               "bitsandbytes>=0.43.0",
               "huggingface_hub>=0.22.0",
               "pandas>=2.0.0",
               "torch>=2.2.0"], check=True)

print("✅ Installation terminée. Redémarrage du kernel requis.")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


✅ Installation terminée. Redémarrage du kernel requis.


{'status': 'ok', 'restart': True}

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Vérification GPU + versions                     ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, numpy as np, torch, transformers, peft, accelerate, datasets

print("="*60)
print("  ENVIRONNEMENT COLAB — V12")
print("="*60)
print(f"  Python       : {sys.version.split()[0]}")
print(f"  NumPy        : {np.__version__}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  PEFT         : {peft.__version__}")
print(f"  Accelerate   : {accelerate.__version__}")
print(f"  Datasets     : {datasets.__version__}")
print("="*60)
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    total_gb = dev.total_memory / 1e9
    free_gb  = torch.cuda.mem_get_info()[0] / 1e9
    print(f"  GPU          : {dev.name}")
    print(f"  VRAM totale  : {total_gb:.1f} GB")
    print(f"  VRAM libre   : {free_gb:.1f} GB")
    print("  ✅ GPU disponible")
else:
    print("  ⚠️  Pas de GPU — entraînement impossible")
print("="*60)


  ENVIRONNEMENT COLAB — V12
  Python       : 3.12.13
  NumPy        : 2.0.2
  PyTorch      : 2.10.0+cu128
  Transformers : 5.0.0
  PEFT         : 0.18.1
  Accelerate   : 1.13.0
  Datasets     : 4.0.0
  GPU          : Tesla T4
  VRAM totale  : 15.6 GB
  VRAM libre   : 15.5 GB
  ✅ GPU disponible


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Téléchargement TinyLlama-1.1B-Chat-v1.0         ║
# ╚══════════════════════════════════════════════════════════════╝
from huggingface_hub import snapshot_download
import shutil, os

MODEL_DIR = "TinyLlama-1.1B-Chat-v1.0"

if not os.path.exists(MODEL_DIR):
    print("📥 Téléchargement TinyLlama-1.1B-Chat-v1.0 (~2.2 GB)...")
    snapshot_download(repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                      local_dir=MODEL_DIR, local_dir_use_symlinks=False)
    print("✅ Modèle téléchargé.")
else:
    print(f"✅ Modèle déjà présent : {MODEL_DIR}")

if not os.path.exists(MODEL_DIR + ".zip"):
    shutil.make_archive(MODEL_DIR, "zip", MODEL_DIR)
    print(f"📦 Archive : {MODEL_DIR}.zip")


📥 Téléchargement TinyLlama-1.1B-Chat-v1.0 (~2.2 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Modèle téléchargé.
📦 Archive : TinyLlama-1.1B-Chat-v1.0.zip


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Table ORACLE_AUDIT_TRAIL (500 lignes simulées)  ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd
from datetime import datetime, timedelta

random.seed(42)

OS_USERS  = ["oracle","sys","system","dbadmin","appuser","reports_user",
             "audit_user","dba_ops","backup_agent","monitor_svc"]
DB_USERS  = ["SYS","SYSTEM","SCOTT","HR","APP_USER","REPORT_USR",
             "AUDIT_USR","DBA_OPS","BACKUP_USR","MONITOR","VROMUALD",
             "ETL_USER","SMART2DADMIN","SMART2DADMINI","TEST"]
SCHEMAS   = ["SYS","HR","SCOTT","APP_SCHEMA","AUDIT_SCHEMA","SMART2DSECU"]
OBJECTS   = ["EMP","DEPT","SALGRADE","FACTURE","CLIENT","TRANSACTION",
             "AUDIT_LOG","V$SESSION","V$SQL","DBA_USERS","DBA_OBJECTS",
             "ALL_TABLES","PAYROLL","EMPLOYEES","CONTRACTS","ORDERS",
             "ACCOUNTS","PRODUCTS","SUPPLIERS","INVOICES","PAYMENTS",
             "SESSIONS","ALERTS","USERS","JOURNAL","BUDGET","AUD$"]
ACTIONS   = ["LOGON","LOGOFF","SELECT","INSERT","UPDATE","DELETE",
             "CREATE TABLE","DROP TABLE","ALTER TABLE","CREATE USER",
             "DROP USER","ALTER USER","GRANT","REVOKE","EXECUTE"]
HOSTS     = ["srv-oracle-01","app-server-01","srv-oracle-02","monitor-01",
             "workstation-12","laptop-rh","srv-etl-03","backup-srv"]
AUTH_TYPES = ["PASSWORD","KERBEROS","CERTIFICATE","NONE"]

base_dt = datetime(2025, 10, 1)
rows = []
for i in range(500):
    dt = base_dt + timedelta(
        days=random.randint(0, 180),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    action = random.choice(ACTIONS)
    obj = random.choice(OBJECTS) if action not in ("LOGON","LOGOFF") else None
    rows.append({
        "ID"                  : i + 1,
        "AUDIT_TYPE"          : random.choice(["STANDARD","UNIFIED"]),
        "SESSIONID"           : random.randint(100000, 999999),
        "OS_USERNAME"         : random.choice(OS_USERS),
        "USERHOST"            : random.choice(HOSTS),
        "TERMINAL"            : f"pts/{random.randint(0,9)}",
        "AUTHENTICATION_TYPE" : random.choice(AUTH_TYPES),
        "DBUSERNAME"          : random.choice(DB_USERS),
        "CLIENT_PROGRAM_NAME" : random.choice(["sqlplus","JDBC","Python","ODI","APEX"]),
        "OBJECT_SCHEMA"       : random.choice(SCHEMAS),
        "OBJECT_NAME"         : obj,
        "SQL_TEXT"            : f"/* sim */ SELECT * FROM {obj};" if obj else None,
        "SQL_BINDS"           : None,
        "EVENT_TIMESTAMP"     : dt.strftime("%Y-%m-%dT%H:%M:%S.%f"),
        "ACTION_NAME"         : action,
        "RETURNCODE"          : 0 if random.random() > 0.1 else random.choice([1017,1031,904]),
        "INSTANCE"            : 1,
    })

AUDIT_DF = pd.DataFrame(rows)
AUDIT_DF.to_csv("oracle_audit_trail.csv", index=False)
print(f"✅ oracle_audit_trail.csv généré : {len(AUDIT_DF)} lignes")
print(AUDIT_DF[["ID","DBUSERNAME","ACTION_NAME","OBJECT_NAME","EVENT_TIMESTAMP"]].head(5).to_string(index=False))


✅ oracle_audit_trail.csv généré : 500 lignes
 ID DBUSERNAME  ACTION_NAME OBJECT_NAME            EVENT_TIMESTAMP
  1        SYS       UPDATE   V$SESSION 2026-03-13T03:01:47.000000
  2   ETL_USER       INSERT      ALERTS 2026-02-07T19:01:35.000000
  3 BACKUP_USR CREATE TABLE     FACTURE 2025-11-25T10:06:05.000000
  4     SYSTEM       DELETE   SUPPLIERS 2026-03-10T19:56:55.000000
  5 BACKUP_USR       SELECT  ALL_TABLES 2026-01-25T20:53:23.000000


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Dataset V9 de base (9 000 exemples)             ║
# ║  Conservé intégralement — base éprouvée depuis V9            ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd

random.seed(42)

TABLE  = "ORACLE_AUDIT_TRAIL"
TS     = "EVENT_TIMESTAMP"
USER_C = "DBUSERNAME"
OBJ_C  = "OBJECT_NAME"
ACT_C  = "ACTION_NAME"
HH24   = f"TO_NUMBER(TO_CHAR({TS},'HH24'))"

USERS = ["SYSTEM","VROMUALD","SYS","HR","APP_USER","REPORT_USR",
         "AUDIT_USR","DBA_OPS","BACKUP_USR","SCOTT","ETL_USER","MONITOR",
         "SMART2DADMIN","TEST"]
OBJECTS = [
    "EMPLOYEES","CLIENTS","TRANSACTIONS","ORDERS","ACCOUNTS",
    "PAYROLL","JOURNAL","CONTRACTS","BUDGET","PRODUCTS","SUPPLIERS",
    "INVOICES","PAYMENTS","AUDIT_LOG","SESSIONS","ALERTS","USERS",
    "AUD$","DBA_USERS","EMP","DEPT"
]

def pad(bloc, target, pool=None):
    pool = pool or bloc
    pfxs = ["Audit : ","Question : ","Analyse : ","Rapport : ",
            "Verifie : ","Securite : ","Oracle : ","Bilan : "]
    while len(bloc) < target:
        ex  = random.choice(pool)
        pfx = random.choice(pfxs)
        q   = ex["instruction"]
        if not any(q.startswith(p) for p in pfxs):
            bloc.append({"instruction": pfx + q[0].lower() + q[1:],
                         "output": ex["output"]})
        else:
            bloc.append(ex)
    return bloc[:target]

# ── BLOC 1 — DBA_USERS (500 ex) ───────────────────────────────
bloc1 = []
b1 = [
    ("Combien d'utilisateurs existent dans la base Oracle ?",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("C'est combien d'utilisateurs en tout ?",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("Y en a combien des comptes Oracle ?",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("Combien y a-t-il de comptes dans la base ?",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("Quels utilisateurs ont un compte Oracle ?",
     f"SELECT USERNAME, ACCOUNT_STATUS FROM DBA_USERS ORDER BY USERNAME;"),
    ("Liste les utilisateurs actifs.",
     f"SELECT USERNAME FROM DBA_USERS WHERE ACCOUNT_STATUS='OPEN' ORDER BY USERNAME;"),
    ("Il y a des comptes bloques ?",
     f"SELECT USERNAME FROM DBA_USERS WHERE ACCOUNT_STATUS='LOCKED' ORDER BY USERNAME;"),
    ("Qui a ete cree ce mois-ci ?",
     f"SELECT USERNAME, CREATED FROM DBA_USERS WHERE TRUNC(CREATED,'MM')=TRUNC(SYSDATE,'MM') ORDER BY CREATED DESC;"),
]
for q, sql in b1:
    bloc1.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Analyse : ","Rapport : "]:
        bloc1.append({"instruction": pfx + q, "output": sql})
bloc1 = pad(bloc1, 500)

# ── BLOC 2 — Utilisateur le plus actif / connexions (600 ex) ──
bloc2 = []
b2 = [
    ("Quel utilisateur a le plus d'actions ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("C'est qui le plus actif ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Top 5 des plus actifs.",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;"),
    ("Qui se connecte le plus ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Combien de connexions LOGON au total ?",
     f"SELECT COUNT(*) AS NB_CONNEXIONS FROM {TABLE} WHERE {ACT_C}='LOGON';"),
    ("Classe les utilisateurs par activite.",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} GROUP BY {USER_C} ORDER BY NB DESC;"),
]
for q, sql in b2:
    bloc2.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Analyse : "]:
        bloc2.append({"instruction": pfx + q, "output": sql})
bloc2 = pad(bloc2, 600)

# ── BLOC 3 — Dernière action / dernier utilisateur (400 ex) ───
bloc3 = []
b3 = [
    ("Quelle est la derniere action enregistree ?",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} ORDER BY {TS} DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Qui a agi en dernier ?",
     f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} ORDER BY {TS} DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Donne-moi les 10 dernieres actions.",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} ORDER BY {TS} DESC FETCH FIRST 10 ROWS ONLY;"),
    ("Quelle est la derniere personne a avoir touche EMPLOYEES ?",
     f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} WHERE {OBJ_C}='EMPLOYEES' ORDER BY {TS} DESC FETCH FIRST 1 ROWS ONLY;"),
]
for q, sql in b3:
    bloc3.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : "]:
        bloc3.append({"instruction": pfx + q, "output": sql})
bloc3 = pad(bloc3, 400)

# ── BLOC 4 — Filtrage par objet (500 ex) ──────────────────────
bloc4 = []
for obj in OBJECTS:
    pairs = [
        (f"Qui a touche {obj} ?",
         f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} WHERE {OBJ_C}='{obj}' ORDER BY {TS} DESC;"),
        (f"Quelles actions sur {obj} ?",
         f"SELECT {ACT_C}, COUNT(*) AS NB FROM {TABLE} WHERE {OBJ_C}='{obj}' GROUP BY {ACT_C} ORDER BY NB DESC;"),
        (f"Combien d'actions sur {obj} ?",
         f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {OBJ_C}='{obj}';"),
    ]
    for q, sql in pairs:
        bloc4.append({"instruction": q, "output": sql})
        for pfx in ["Audit : ","Question : "]:
            bloc4.append({"instruction": pfx + q, "output": sql})
bloc4 = pad(bloc4, 500)

# ── BLOC 5 — Filtrage par utilisateur (500 ex) ────────────────
bloc5 = []
for user in USERS:
    pairs = [
        (f"Que fait {user} ?",
         f"SELECT {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {USER_C}='{user}' ORDER BY {TS} DESC;"),
        (f"Quelles actions de {user} ?",
         f"SELECT {ACT_C}, COUNT(*) AS NB FROM {TABLE} WHERE {USER_C}='{user}' GROUP BY {ACT_C} ORDER BY NB DESC;"),
        (f"Combien d'actions a {user} effectuees ?",
         f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {USER_C}='{user}';"),
    ]
    for q, sql in pairs:
        bloc5.append({"instruction": q, "output": sql})
        for pfx in ["Audit : ","Question : "]:
            bloc5.append({"instruction": pfx + q, "output": sql})
bloc5 = pad(bloc5, 500)

# ── BLOC 6 — Sécurité / erreurs / RETURNCODE (400 ex) ─────────
bloc6 = []
b6 = [
    ("Y a-t-il eu des echecs de connexion ?",
     f"SELECT {USER_C}, {TS}, RETURNCODE FROM {TABLE} WHERE {ACT_C}='LOGON' AND RETURNCODE!=0 ORDER BY {TS} DESC;"),
    ("Quels evenements se sont soldes par une erreur ?",
     f"SELECT {USER_C}, {ACT_C}, RETURNCODE, {TS} FROM {TABLE} WHERE RETURNCODE!=0 ORDER BY {TS} DESC;"),
    ("Supprime tous les enregistrements.",
     "-- REQUETE REFUSEE : les operations DELETE ne sont pas autorisees."),
    ("Fais un INSERT dans la table.",
     "-- REQUETE REFUSEE : les operations INSERT ne sont pas autorisees."),
    ("DROP TABLE EMPLOYEES",
     "-- REQUETE REFUSEE : les operations DDL ne sont pas autorisees."),
    ("Qui a essaye de se connecter sans succes ?",
     f"SELECT DISTINCT {USER_C} FROM {TABLE} WHERE {ACT_C}='LOGON' AND RETURNCODE=1017;"),
]
for q, sql in b6:
    bloc6.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Securite : "]:
        bloc6.append({"instruction": pfx + q, "output": sql})
bloc6 = pad(bloc6, 400)

# ── BLOC 7 — Filtrage temporel SYSDATE (600 ex) ───────────────
bloc7 = []
for n in [1, 2, 3, 7, 14, 30, 60, 90]:
    b7 = [
        (f"Activite des {n} derniers jours ?",
         f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE TRUNC({TS})>=TRUNC(SYSDATE-{n}) ORDER BY {TS} DESC;"),
        (f"Qui s'est connecte dans les {n} derniers jours ?",
         f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})>=TRUNC(SYSDATE-{n}) ORDER BY {TS} DESC;"),
        (f"Quelles modifications dans les {n} derniers jours ?",
         f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') AND TRUNC({TS})>=TRUNC(SYSDATE-{n}) ORDER BY {TS} DESC;"),
    ]
    for q, sql in b7:
        bloc7.append({"instruction": q, "output": sql})
        for pfx in ["Audit : ","Question : "]:
            bloc7.append({"instruction": pfx + q, "output": sql})
bloc7 = pad(bloc7, 600)

# ── BLOC 8 — Actions DDL/DCL (400 ex) ─────────────────────────
bloc8 = []
b8 = [
    ("Y a-t-il eu des creations de tables ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='CREATE TABLE' ORDER BY {TS} DESC;"),
    ("Y a-t-il eu des suppressions de tables ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='DROP TABLE' ORDER BY {TS} DESC;"),
    ("Y a-t-il eu des creations de comptes ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='CREATE USER' ORDER BY {TS} DESC;"),
    ("Y a-t-il eu des suppressions de comptes utilisateurs ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='DROP USER' ORDER BY {TS} DESC;"),
    ("Y a-t-il eu des modifications de comptes ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='ALTER USER' ORDER BY {TS} DESC;"),
    ("Quels privileges ont ete accordes ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='GRANT' ORDER BY {TS} DESC;"),
    ("Quels privileges ont ete retires ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='REVOKE' ORDER BY {TS} DESC;"),
]
for q, sql in b8:
    bloc8.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Securite : "]:
        bloc8.append({"instruction": pfx + q, "output": sql})
bloc8 = pad(bloc8, 400)

# ── BLOC 9 — COUNT DISTINCT + agrégats complexes (400 ex) ─────
bloc9 = []
b9 = [
    ("Combien de tables differentes ont ete touchees ?",
     f"SELECT COUNT(DISTINCT {OBJ_C}) AS NB_TABLES FROM {TABLE} WHERE {OBJ_C} IS NOT NULL;"),
    ("Combien d'utilisateurs distincts ont agi ?",
     f"SELECT COUNT(DISTINCT {USER_C}) AS NB_USERS FROM {TABLE};"),
    ("Quels sont les 3 utilisateurs ayant modifie le plus de tables differentes ?",
     f"SELECT {USER_C}, COUNT(DISTINCT {OBJ_C}) AS NB_TABLES FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') GROUP BY {USER_C} ORDER BY NB_TABLES DESC FETCH FIRST 3 ROWS ONLY;"),
    ("Quelle table a ete consultee le plus souvent ?",
     f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='SELECT' AND {OBJ_C} IS NOT NULL GROUP BY {OBJ_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Quelle table a ete la plus modifiee ?",
     f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') AND {OBJ_C} IS NOT NULL GROUP BY {OBJ_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
]
for q, sql in b9:
    bloc9.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Analyse : "]:
        bloc9.append({"instruction": pfx + q, "output": sql})
bloc9 = pad(bloc9, 400)

# ── BLOC 10 — Activité nocturne / horaires (300 ex) ───────────
bloc10 = []
b10 = [
    ("Y a-t-il de l'activite apres 22h ?",
     f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} WHERE {HH24}>=22 ORDER BY {TS} DESC;"),
    ("Quels utilisateurs se sont connectes entre 22h et 6h ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND ({HH24}>=22 OR {HH24}<6) ORDER BY {TS} DESC;"),
    ("Activite suspecte la nuit ?",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE ({HH24}>=22 OR {HH24}<6) ORDER BY {TS} DESC;"),
    ("Qui a agi avant 6h du matin ?",
     f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} WHERE {HH24}<6 ORDER BY {TS} DESC;"),
]
for q, sql in b10:
    bloc10.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Securite : "]:
        bloc10.append({"instruction": pfx + q, "output": sql})
bloc10 = pad(bloc10, 300)

# ── BLOC 11 — Questions vagues / orales (500 ex) ──────────────
bloc11 = []
b11 = [
    ("Montre-moi tout.",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} ORDER BY {TS} DESC FETCH FIRST 50 ROWS ONLY;"),
    ("Quoi de neuf ?",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE TRUNC({TS})>=TRUNC(SYSDATE-7) ORDER BY {TS} DESC;"),
    ("Quelqu'un a fait quelque chose hier ?",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Resume les activites recentes.",
     f"SELECT {USER_C}, {ACT_C}, COUNT(*) AS NB FROM {TABLE} WHERE TRUNC({TS})>=TRUNC(SYSDATE-7) GROUP BY {USER_C}, {ACT_C} ORDER BY NB DESC;"),
    ("C'est calme ou actif aujourd'hui ?",
     f"SELECT COUNT(*) AS NB_ACTIONS FROM {TABLE} WHERE TRUNC({TS})=TRUNC(SYSDATE);"),
]
for q, sql in b11:
    bloc11.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Bilan : "]:
        bloc11.append({"instruction": pfx + q, "output": sql})
bloc11 = pad(bloc11, 500)

# ── BLOC 12 — DML spécifiques (400 ex) ────────────────────────
bloc12 = []
b12 = [
    ("Qui a fait des SELECT ?",
     f"SELECT DISTINCT {USER_C} FROM {TABLE} WHERE {ACT_C}='SELECT' ORDER BY {USER_C};"),
    ("Qui a fait des INSERT ?",
     f"SELECT DISTINCT {USER_C} FROM {TABLE} WHERE {ACT_C}='INSERT' ORDER BY {USER_C};"),
    ("Qui a fait des UPDATE ?",
     f"SELECT DISTINCT {USER_C} FROM {TABLE} WHERE {ACT_C}='UPDATE' ORDER BY {USER_C};"),
    ("Qui a fait des DELETE ?",
     f"SELECT DISTINCT {USER_C} FROM {TABLE} WHERE {ACT_C}='DELETE' ORDER BY {USER_C};"),
    ("Quelles tables ont ete lues ?",
     f"SELECT DISTINCT {OBJ_C} FROM {TABLE} WHERE {ACT_C}='SELECT' AND {OBJ_C} IS NOT NULL ORDER BY {OBJ_C};"),
]
for q, sql in b12:
    bloc12.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : "]:
        bloc12.append({"instruction": pfx + q, "output": sql})
bloc12 = pad(bloc12, 400)

# ── ASSEMBLAGE V9 ──────────────────────────────────────────────
all_examples_v9 = (bloc1 + bloc2 + bloc3 + bloc4 + bloc5 +
                   bloc6 + bloc7 + bloc8 + bloc9 + bloc10 +
                   bloc11 + bloc12)

import random
random.seed(42)
random.shuffle(all_examples_v9)
all_examples_v9 = all_examples_v9[:9000]

print(f"✅ Dataset V9 assemblé : {len(all_examples_v9)} exemples")
blocs = {"B1-DBA_USERS":500,"B2-Actif":600,"B3-Dernier":400,"B4-Objet":500,
         "B5-User":500,"B6-Secu":400,"B7-Temporel":600,"B8-DDL":400,
         "B9-Agreg":400,"B10-Nuit":300,"B11-Vague":500,"B12-DML":400}
print("  Blocs :", {k:v for k,v in blocs.items()})


✅ Dataset V9 assemblé : 5500 exemples
  Blocs : {'B1-DBA_USERS': 500, 'B2-Actif': 600, 'B3-Dernier': 400, 'B4-Objet': 500, 'B5-User': 500, 'B6-Secu': 400, 'B7-Temporel': 600, 'B8-DDL': 400, 'B9-Agreg': 400, 'B10-Nuit': 300, 'B11-Vague': 500, 'B12-DML': 400}


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Patch V10 : Blocs C-F                           ║
# ║  Mois nommés · GROUP BY OBJECT_NAME · Actions contextuelles  ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd

random.seed(42)

MONTHS_FR = {
    "janvier":1,"fevrier":2,"mars":3,"avril":4,"mai":5,"juin":6,
    "juillet":7,"aout":8,"septembre":9,"octobre":10,"novembre":11,"decembre":12
}

def month_range_sql(month_num, year=2026):
    start = f"{year}-{month_num:02d}-01"
    next_start = f"{year+1}-01-01" if month_num==12 else f"{year}-{month_num+1:02d}-01"
    return (f"TRUNC({TS}) >= TO_DATE('{start}','YYYY-MM-DD') "
            f"AND TRUNC({TS}) < TO_DATE('{next_start}','YYYY-MM-DD')")

# ── BLOC C — Mois nommés (500 ex) ─────────────────────────────
blocC = []
for m_name, m_num in MONTHS_FR.items():
    cond = month_range_sql(m_num, 2026)
    pairs = [
        (f"Qui s'est connecte en {m_name} 2026 ?",
         f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND {cond} ORDER BY {TS} DESC;"),
        (f"Qu'est-ce qui s'est passe en {m_name} 2026 ?",
         f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {cond} ORDER BY {TS} DESC;"),
        (f"Combien d'actions en {m_name} 2026 ?",
         f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {cond};"),
        (f"Quelle table a ete la plus touchee en {m_name} 2026 ?",
         f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {OBJ_C} IS NOT NULL AND {cond} GROUP BY {OBJ_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ]
    for q, sql in pairs:
        blocC.append({"instruction": q, "output": sql})
        for pfx in ["Audit : ","Question : ","Analyse : "]:
            blocC.append({"instruction": pfx + q, "output": sql})

# Relatifs
rel_c = [
    ("Actions du mois dernier ?",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE TRUNC({TS},'MM')=TRUNC(ADD_MONTHS(SYSDATE,-1),'MM') ORDER BY {TS} DESC;"),
    ("Connexions il y a 2 mois ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS},'MM')=TRUNC(ADD_MONTHS(SYSDATE,-2),'MM') ORDER BY {TS} DESC;"),
]
for q, sql in rel_c:
    blocC.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : "]:
        blocC.append({"instruction": pfx + q, "output": sql})

while len(blocC) < 500:
    ex = random.choice(blocC)
    blocC.append(ex)
blocC = blocC[:500]

# ── BLOC D — GROUP BY OBJECT_NAME (400 ex) ────────────────────
blocD = []
pairs_d = [
    ("Quelle table a ete la plus modifiee ?",
     f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') AND {OBJ_C} IS NOT NULL GROUP BY {OBJ_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Les 5 objets les plus touches ?",
     f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {OBJ_C} IS NOT NULL GROUP BY {OBJ_C} ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;"),
    ("Quelles tables ont ete modifiees cette semaine ?",
     f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') AND {OBJ_C} IS NOT NULL AND TRUNC({TS})>=TRUNC(SYSDATE-7) GROUP BY {OBJ_C} ORDER BY NB DESC;"),
    ("Quels objets ont change ces 30 derniers jours ?",
     f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') AND {OBJ_C} IS NOT NULL AND TRUNC({TS})>=TRUNC(SYSDATE-30) GROUP BY {OBJ_C} ORDER BY NB DESC;"),
]
for q, sql in pairs_d:
    blocD.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Analyse : ","Rapport : "]:
        blocD.append({"instruction": pfx + q, "output": sql})
while len(blocD) < 400:
    blocD.append(random.choice(blocD))
blocD = blocD[:400]

# ── BLOC E — ACTION_NAME contextualisé (300 ex) ───────────────
blocE = []
pairs_e = [
    ("Qui a accede a EMPLOYEES ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {OBJ_C}='EMPLOYEES' AND {ACT_C} IN ('SELECT','INSERT','UPDATE','DELETE') GROUP BY {USER_C} ORDER BY NB DESC;"),
    ("Qui a modifie des donnees dans PAYROLL ?",
     f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} WHERE {OBJ_C}='PAYROLL' AND {ACT_C} IN ('INSERT','UPDATE','DELETE') ORDER BY {TS} DESC;"),
    ("Y a-t-il eu des creations de comptes ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='CREATE USER' ORDER BY {TS} DESC;"),
    ("Y a-t-il eu des suppressions de comptes utilisateurs ?",
     f"SELECT {USER_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C}='DROP USER' ORDER BY {TS} DESC;"),
    ("Qui a lu la table ACCOUNTS ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {OBJ_C}='ACCOUNTS' AND {ACT_C}='SELECT' GROUP BY {USER_C} ORDER BY NB DESC;"),
]
for q, sql in pairs_e:
    blocE.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Securite : "]:
        blocE.append({"instruction": pfx + q, "output": sql})
while len(blocE) < 300:
    blocE.append(random.choice(blocE))
blocE = blocE[:300]

# ── BLOC F — Langage métier (200 ex) ──────────────────────────
blocF = []
pairs_f = [
    ("Qui a le plus change d'informations ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Quel poste a ete le plus actif ?",
     f"SELECT USERHOST, COUNT(*) AS NB FROM {TABLE} WHERE USERHOST IS NOT NULL GROUP BY USERHOST ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Depuis quelle machine se fait le plus de connexions ?",
     f"SELECT USERHOST, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' AND USERHOST IS NOT NULL GROUP BY USERHOST ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Qui a le plus consulte de donnees ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='SELECT' GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
]
for q, sql in pairs_f:
    blocF.append({"instruction": q, "output": sql})
    for pfx in ["Audit : ","Question : ","Analyse : "]:
        blocF.append({"instruction": pfx + q, "output": sql})
while len(blocF) < 200:
    blocF.append(random.choice(blocF))
blocF = blocF[:200]

all_examples_v10 = all_examples_v9 + blocC + blocD + blocE + blocF
print(f"✅ Dataset V10 : {len(all_examples_v10)} exemples")
print(f"   + BlocC(mois):{len(blocC)} + BlocD(obj):{len(blocD)} + BlocE(actions):{len(blocE)} + BlocF(metier):{len(blocF)}")


✅ Dataset V10 : 6900 exemples
   + BlocC(mois):500 + BlocD(obj):400 + BlocE(actions):300 + BlocF(metier):200


In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Patch V11 : Blocs G-H                           ║
# ║  Temporel paramétrique massif · Error-driven                  ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd

random.seed(42)

blocG = []
prefixes = ["Question : ","Audit : ","Analyse : "]

# Formes absolues
absolutes = [
    ("hier",          f"TRUNC({TS})=TRUNC(SYSDATE-1)"),
    ("avant hier",    f"TRUNC({TS})=TRUNC(SYSDATE-2)"),
    ("aujourd hui",   f"TRUNC({TS})=TRUNC(SYSDATE)"),
    ("cette semaine", f"TRUNC({TS},'IW')=TRUNC(SYSDATE,'IW')"),
    ("ce mois",       f"TRUNC({TS},'MM')=TRUNC(SYSDATE,'MM')"),
]
for label, cond in absolutes:
    for q_tpl, sql_tpl in [
        (f"Qu est ce qui s est passe {label} ?",
         f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {cond} ORDER BY {TS} DESC;"),
        (f"Qui s est connecte {label} ?",
         f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND {cond} ORDER BY {TS} DESC;"),
    ]:
        blocG.append({"instruction": q_tpl, "output": sql_tpl})
        for pfx in prefixes:
            blocG.append({"instruction": pfx + q_tpl, "output": sql_tpl})

# Fenêtres N jours
for n in [1,2,3,5,7,10,14,21,30,45,60,90,120,180]:
    cond = f"TRUNC({TS})>=TRUNC(SYSDATE-{n})"
    for phrase in [f"les {n} derniers jours", f"durant les {n} derniers jours",
                   f"sur les {n} derniers jours", f"ces {n} derniers jours"]:
        pairs = [
            (f"Qu est ce qui s est passe {phrase} ?",
             f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {cond} ORDER BY {TS} DESC;"),
            (f"Qui s est connecte {phrase} ?",
             f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND {cond} ORDER BY {TS} DESC;"),
            (f"Quelles modifications {phrase} ?",
             f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {ACT_C} IN ('INSERT','UPDATE','DELETE') AND {cond} ORDER BY {TS} DESC;"),
        ]
        for q, sql in pairs:
            blocG.append({"instruction": q, "output": sql})
            for pfx in prefixes:
                blocG.append({"instruction": pfx + q, "output": sql})

# Nocturne
night_cond = f"({HH24}>=22 OR {HH24}<6)"
for q, sql in [
    ("Quels utilisateurs se sont connectes entre 22h et 6h du matin ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND {night_cond} ORDER BY {TS} DESC;"),
    ("Qui a agi la nuit ?",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {night_cond} ORDER BY {TS} DESC;"),
]:
    blocG.append({"instruction": q, "output": sql})
    for pfx in prefixes:
        blocG.append({"instruction": pfx + q, "output": sql})

print(f"blocG généré : {len(blocG)} exemples")

# ── BLOC H — Error-driven V11 (cas réels) ─────────────────────
blocH = []
error_cases = [
    ("liste les 5 objets les plus consultes au cours des 30 derniers jours",
     f"SELECT {OBJ_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='SELECT' AND {OBJ_C} IS NOT NULL AND {TS}>=SYSDATE-30 GROUP BY {OBJ_C} ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;"),
    ("quels utilisateurs se sont connectes entre 22h et 6h du matin ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND ({HH24}>=22 OR {HH24}<6) ORDER BY {TS} DESC;"),
    ("y a t il un utilisateur qui s est connecte durant les deux dernieres semaines ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})>=TRUNC(SYSDATE-14) ORDER BY {TS} DESC;"),
    ("qu est ce qui s est passe dans la base hier ?",
     f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
]
for q, sql in error_cases:
    for pfx in ["", "Question : ", "Audit : "]:
        blocH.append({"instruction": pfx + q, "output": sql})

print(f"blocH généré : {len(blocH)} exemples")
all_examples_v11 = all_examples_v10 + blocG + blocH
print(f"✅ Dataset V11 cumulé : {len(all_examples_v11)} exemples")


blocG généré : 720 exemples
blocH généré : 12 exemples
✅ Dataset V11 cumulé : 7632 exemples


In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — NOUVEAU V12 : Blocs I-L                         ║
# ║                                                               ║
# ║  Faille α : ACTION_NAME='LOGON' toujours explicite           ║
# ║  Faille β : heures → SYSDATE-N/24 ou /1440                  ║
# ║  Faille δ : double filtre acteur + objet                     ║
# ║  Faille ε : USERHOST depuis "poste/machine/terminal"         ║
# ║  Enrichissement : formulations orales · synonymes · variantes║
# ╚══════════════════════════════════════════════════════════════╝
import random
random.seed(42)

# ──────────────────────────────────────────────────────────────
# BLOC I — Faille α : ACTION_NAME='LOGON' toujours complet (600 ex)
# Le modèle générait "WHERE LOGON" sans "ACTION_NAME="
# ──────────────────────────────────────────────────────────────
blocI = []

logon_questions = [
    # Hier
    ("Qui s'est connecte hier ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Montre-moi les connexions d'hier.",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Quelles connexions ont eu lieu hier ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Y a-t-il eu des connexions hier ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Connexions d'hier s'il te plait.",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    # Aujourd'hui
    ("Qui s'est connecte aujourd'hui ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE) ORDER BY {TS} DESC;"),
    ("Connexions du jour.",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE) ORDER BY {TS} DESC;"),
    # Cette semaine
    ("Qui s'est connecte cette semaine ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS},'IW')=TRUNC(SYSDATE,'IW') ORDER BY {TS} DESC;"),
    # Ce mois
    ("Qui s'est connecte ce mois-ci ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS},'MM')=TRUNC(SYSDATE,'MM') ORDER BY {TS} DESC;"),
    # Derniers N jours
    ("Qui s'est connecte dans les 7 derniers jours ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})>=TRUNC(SYSDATE-7) ORDER BY {TS} DESC;"),
    ("Qui s'est connecte dans les 14 derniers jours ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})>=TRUNC(SYSDATE-14) ORDER BY {TS} DESC;"),
    ("Qui s'est connecte dans les 30 derniers jours ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})>=TRUNC(SYSDATE-30) ORDER BY {TS} DESC;"),
    # Comptage LOGON
    ("Combien de connexions hier ?",
     f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1);"),
    ("Combien de connexions ce mois-ci ?",
     f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS},'MM')=TRUNC(SYSDATE,'MM');"),
    ("Combien de fois VROMUALD s'est connecte ?",
     f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' AND {USER_C}='VROMUALD';"),
    ("Combien de fois SYSTEM s'est connecte ce mois ?",
     f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' AND {USER_C}='SYSTEM' AND TRUNC({TS},'MM')=TRUNC(SYSDATE,'MM');"),
    # Qui se connecte le plus
    ("Quel utilisateur s'est connecte le plus ?",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Top 3 des connexions.",
     f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' GROUP BY {USER_C} ORDER BY NB DESC FETCH FIRST 3 ROWS ONLY;"),
    # Première/dernière connexion
    ("Qui s'est connecte en premier aujourd'hui ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE) ORDER BY {TS} ASC FETCH FIRST 1 ROWS ONLY;"),
    ("Derniere connexion enregistree ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' ORDER BY {TS} DESC FETCH FIRST 1 ROWS ONLY;"),
]

for q, sql in logon_questions:
    blocI.append({"instruction": q, "output": sql})
    for pfx in ["Question : ","Audit : ","Analyse : ","Rapport : ","Securite : "]:
        blocI.append({"instruction": pfx + q, "output": sql})

# Variantes orales supplémentaires
logon_extra = [
    ("Donne-moi les gens qui se sont connectes hier.",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Est-ce que quelqu'un s'est connecte hier ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Dis-moi qui s'est connecte hier.",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Liste des connexions d'hier.",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
    ("Qui etait connecte hier ?",
     f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND TRUNC({TS})=TRUNC(SYSDATE-1) ORDER BY {TS} DESC;"),
]
for q, sql in logon_extra:
    blocI.append({"instruction": q, "output": sql})
    for pfx in ["Question : ","Audit : "]:
        blocI.append({"instruction": pfx + q, "output": sql})

while len(blocI) < 600:
    blocI.append(random.choice(blocI))
blocI = blocI[:600]
print(f"Bloc I (α LOGON explicite) : {len(blocI)} exemples")

# ──────────────────────────────────────────────────────────────
# BLOC J — Faille β : fenêtres en heures → SYSDATE-N/24 (400 ex)
# ──────────────────────────────────────────────────────────────
blocJ = []

# Paires heures → formule correcte Oracle
hours_pairs = []
for h in [1, 2, 3, 4, 6, 8, 12, 24, 36, 48, 72]:
    days_frac = f"SYSDATE-{h}/24"
    for phrase_q, phrase_sql in [
        (f"les {h} dernieres heures", f"{TS}>={days_frac}"),
        (f"ces {h} dernieres heures", f"{TS}>={days_frac}"),
        (f"durant les {h} dernieres heures", f"{TS}>={days_frac}"),
        (f"dans les {h} dernieres heures", f"{TS}>={days_frac}"),
    ]:
        hours_pairs.append((
            f"Qui s'est connecte {phrase_q} ?",
            f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND {phrase_sql} ORDER BY {TS} DESC;"
        ))
        hours_pairs.append((
            f"Qu'est-ce qui s'est passe {phrase_q} ?",
            f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {phrase_sql} ORDER BY {TS} DESC;"
        ))
        hours_pairs.append((
            f"Quelles actions {phrase_q} ?",
            f"SELECT {USER_C}, {ACT_C}, {OBJ_C}, {TS} FROM {TABLE} WHERE {phrase_sql} ORDER BY {TS} DESC;"
        ))

# Minutes → SYSDATE-N/1440
for m in [15, 30, 45, 60]:
    for phrase in [f"les {m} dernieres minutes", f"dans les {m} dernieres minutes"]:
        hours_pairs.append((
            f"Y a-t-il eu des actions {phrase} ?",
            f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} WHERE {TS}>=SYSDATE-{m}/1440 ORDER BY {TS} DESC;"
        ))

for q, sql in hours_pairs:
    blocJ.append({"instruction": q, "output": sql})
    for pfx in ["Question : ","Audit : ","Analyse : "]:
        blocJ.append({"instruction": pfx + q, "output": sql})

while len(blocJ) < 400:
    blocJ.append(random.choice(blocJ))
blocJ = blocJ[:400]
print(f"Bloc J (β heures→SYSDATE/24) : {len(blocJ)} exemples")

# ──────────────────────────────────────────────────────────────
# BLOC K — Faille δ : double filtre acteur + objet (500 ex)
# ──────────────────────────────────────────────────────────────
blocK = []

for user in USERS:
    for obj in OBJECTS[:10]:  # 10 objets × 14 users = 140 paires
        pairs_k = [
            # Acteur + Objet : toutes actions
            (f"Quelles actions {user} a-t-il effectuees sur {obj} ?",
             f"SELECT {ACT_C}, {TS} FROM {TABLE} WHERE {USER_C}='{user}' AND {OBJ_C}='{obj}' ORDER BY {TS} DESC;"),
            # Acteur + Objet : modifications
            (f"Est-ce que {user} a modifie {obj} ?",
             f"SELECT {ACT_C}, {TS} FROM {TABLE} WHERE {USER_C}='{user}' AND {OBJ_C}='{obj}' AND {ACT_C} IN ('INSERT','UPDATE','DELETE') ORDER BY {TS} DESC;"),
            # Acteur + Objet : COUNT
            (f"Combien de fois {user} a-t-il touche {obj} ?",
             f"SELECT COUNT(*) AS NB FROM {TABLE} WHERE {USER_C}='{user}' AND {OBJ_C}='{obj}';"),
            # Acteur + Objet + période
            (f"Est-ce que {user} a accede a {obj} cette semaine ?",
             f"SELECT {ACT_C}, {TS} FROM {TABLE} WHERE {USER_C}='{user}' AND {OBJ_C}='{obj}' AND TRUNC({TS},'IW')=TRUNC(SYSDATE,'IW') ORDER BY {TS} DESC;"),
        ]
        for q, sql in pairs_k:
            blocK.append({"instruction": q, "output": sql})
            for pfx in ["Question : ","Audit : "]:
                blocK.append({"instruction": pfx + q, "output": sql})

# Variantes orales
oral_k = [
    ("Quelles actions VROMUALD a-t-il effectuees sur la table EMPLOYEES ?",
     f"SELECT {ACT_C}, {TS} FROM {TABLE} WHERE {USER_C}='VROMUALD' AND {OBJ_C}='EMPLOYEES' ORDER BY {TS} DESC;"),
    ("Est-ce que SYSTEM a touche PAYROLL ?",
     f"SELECT {ACT_C}, {TS} FROM {TABLE} WHERE {USER_C}='SYSTEM' AND {OBJ_C}='PAYROLL' ORDER BY {TS} DESC;"),
    ("HR a-t-il fait des SELECT sur EMPLOYEES ?",
     f"SELECT {TS} FROM {TABLE} WHERE {USER_C}='HR' AND {OBJ_C}='EMPLOYEES' AND {ACT_C}='SELECT' ORDER BY {TS} DESC;"),
    ("SCOTT a-t-il modifie DEPT ?",
     f"SELECT {ACT_C}, {TS} FROM {TABLE} WHERE {USER_C}='SCOTT' AND {OBJ_C}='DEPT' AND {ACT_C} IN ('INSERT','UPDATE','DELETE') ORDER BY {TS} DESC;"),
    ("Quand VROMUALD a-t-il accede a CLIENTS pour la derniere fois ?",
     f"SELECT {TS} FROM {TABLE} WHERE {USER_C}='VROMUALD' AND {OBJ_C}='CLIENTS' ORDER BY {TS} DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Est-ce que DBA_OPS a cree des tables recemment ?",
     f"SELECT {OBJ_C}, {TS} FROM {TABLE} WHERE {USER_C}='DBA_OPS' AND {ACT_C}='CREATE TABLE' ORDER BY {TS} DESC;"),
]
for q, sql in oral_k:
    blocK.append({"instruction": q, "output": sql})
    for pfx in ["Question : ","Audit : ","Securite : "]:
        blocK.append({"instruction": pfx + q, "output": sql})

while len(blocK) < 500:
    blocK.append(random.choice(blocK))
blocK = blocK[:500]
print(f"Bloc K (δ double filtre acteur+objet) : {len(blocK)} exemples")

# ──────────────────────────────────────────────────────────────
# BLOC L — Faille ε : USERHOST depuis poste/machine/terminal (300 ex)
# ──────────────────────────────────────────────────────────────
blocL = []

HOSTS_LIST = ["srv-oracle-01","app-server-01","srv-oracle-02","monitor-01",
              "workstation-12","laptop-rh","srv-etl-03","backup-srv"]

# Questions génériques sur USERHOST
userhost_generic = [
    ("Quel poste de travail a effectue le plus de connexions ?",
     f"SELECT USERHOST, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' AND USERHOST IS NOT NULL GROUP BY USERHOST ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Depuis quelle machine y a-t-il le plus d'activite ?",
     f"SELECT USERHOST, COUNT(*) AS NB FROM {TABLE} WHERE USERHOST IS NOT NULL GROUP BY USERHOST ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Quel terminal est le plus utilise ?",
     f"SELECT USERHOST, COUNT(*) AS NB FROM {TABLE} WHERE USERHOST IS NOT NULL GROUP BY USERHOST ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Liste les postes qui se sont connectes.",
     f"SELECT DISTINCT USERHOST FROM {TABLE} WHERE {ACT_C}='LOGON' AND USERHOST IS NOT NULL ORDER BY USERHOST;"),
    ("Quelles machines ont acces a la base ?",
     f"SELECT DISTINCT USERHOST FROM {TABLE} WHERE USERHOST IS NOT NULL ORDER BY USERHOST;"),
    ("D'ou se connectent les utilisateurs ?",
     f"SELECT USERHOST, COUNT(*) AS NB FROM {TABLE} WHERE {ACT_C}='LOGON' AND USERHOST IS NOT NULL GROUP BY USERHOST ORDER BY NB DESC;"),
    ("Top 5 des machines les plus actives.",
     f"SELECT USERHOST, COUNT(*) AS NB FROM {TABLE} WHERE USERHOST IS NOT NULL GROUP BY USERHOST ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;"),
    ("Quels postes ont agi cette semaine ?",
     f"SELECT DISTINCT USERHOST FROM {TABLE} WHERE USERHOST IS NOT NULL AND TRUNC({TS},'IW')=TRUNC(SYSDATE,'IW') ORDER BY USERHOST;"),
    ("Y a-t-il eu des connexions depuis un poste inconnu ?",
     f"SELECT USERHOST, {USER_C}, {TS} FROM {TABLE} WHERE {ACT_C}='LOGON' AND USERHOST IS NOT NULL ORDER BY {TS} DESC;"),
    ("Depuis combien de machines differentes s'est-on connecte ?",
     f"SELECT COUNT(DISTINCT USERHOST) AS NB_MACHINES FROM {TABLE} WHERE {ACT_C}='LOGON' AND USERHOST IS NOT NULL;"),
]
for q, sql in userhost_generic:
    blocL.append({"instruction": q, "output": sql})
    for pfx in ["Question : ","Audit : ","Analyse : ","Securite : ","Rapport : "]:
        blocL.append({"instruction": pfx + q, "output": sql})

# Questions spécifiques par host
for host in HOSTS_LIST:
    pairs_l = [
        (f"Qui s'est connecte depuis {host} ?",
         f"SELECT {USER_C}, {TS} FROM {TABLE} WHERE USERHOST='{host}' AND {ACT_C}='LOGON' ORDER BY {TS} DESC;"),
        (f"Quelles actions depuis {host} ?",
         f"SELECT {USER_C}, {ACT_C}, {TS} FROM {TABLE} WHERE USERHOST='{host}' ORDER BY {TS} DESC;"),
    ]
    for q, sql in pairs_l:
        blocL.append({"instruction": q, "output": sql})
        for pfx in ["Question : ","Audit : "]:
            blocL.append({"instruction": pfx + q, "output": sql})

while len(blocL) < 300:
    blocL.append(random.choice(blocL))
blocL = blocL[:300]
print(f"Bloc L (ε USERHOST poste/machine) : {len(blocL)} exemples")

# ── ASSEMBLAGE V12 ─────────────────────────────────────────────
all_examples_v12_raw = all_examples_v11 + blocI + blocJ + blocK + blocL
print(f"\n✅ Dataset V12 brut : {len(all_examples_v12_raw)} exemples")
print(f"   Blocs V12 ajoutés : I({len(blocI)}) + J({len(blocJ)}) + K({len(blocK)}) + L({len(blocL)})")


Bloc I (α LOGON explicite) : 600 exemples
Bloc J (β heures→SYSDATE/24) : 400 exemples
Bloc K (δ double filtre acteur+objet) : 500 exemples
Bloc L (ε USERHOST poste/machine) : 300 exemples

✅ Dataset V12 brut : 9432 exemples
   Blocs V12 ajoutés : I(600) + J(400) + K(500) + L(300)


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 9 — Fusion + Export dataset V12                     ║
# ║  Rééquilibrage ciblé · Déduplication · Shuffle               ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd
random.seed(42)

TARGET = 9500   # légèrement plus grand que V11 pour meilleure couverture

def to_df(examples, source):
    df = pd.DataFrame(list(examples))
    if df.empty:
        return pd.DataFrame(columns=["instruction","output","source"])
    df["instruction"] = df["instruction"].astype(str).str.strip()
    df["output"]      = df["output"].astype(str).str.strip()
    df["source"]      = source
    return df[["instruction","output","source"]]

# Pools par catégorie
df_base   = to_df(all_examples_v9,   "base")
df_v10    = to_df(blocC + blocD + blocE + blocF, "v10_patch")
df_v11    = to_df(blocG + blocH,      "v11_patch")
df_v12    = to_df(blocI + blocJ + blocK + blocL, "v12_patch")

df_all = pd.concat([df_base, df_v10, df_v11, df_v12], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["instruction","output"], keep="first").reset_index(drop=True)

print(f"Total unique après dédup : {len(df_all)}")
print(f"  base      : {len(df_all[df_all.source=='base'])}")
print(f"  v10_patch : {len(df_all[df_all.source=='v10_patch'])}")
print(f"  v11_patch : {len(df_all[df_all.source=='v11_patch'])}")
print(f"  v12_patch : {len(df_all[df_all.source=='v12_patch'])}")

# Quotas rééquilibrés
# V12 patch = priorité max (failles critiques)
# Base = fondation solide
# V10/V11 = enrichissement
q_v12   = min(int(TARGET * 0.20), len(df_all[df_all.source=='v12_patch']))  # 20%
q_v11   = min(int(TARGET * 0.15), len(df_all[df_all.source=='v11_patch']))  # 15%
q_v10   = min(int(TARGET * 0.15), len(df_all[df_all.source=='v10_patch']))  # 15%
q_base  = TARGET - q_v12 - q_v11 - q_v10                                    # 50%

def sample(df, n, seed):
    if len(df) == 0: return df
    replace = len(df) < n
    return df.sample(n=n, replace=replace, random_state=seed)

s_v12  = sample(df_all[df_all.source=='v12_patch'], q_v12,  42)
s_v11  = sample(df_all[df_all.source=='v11_patch'], q_v11,  43)
s_v10  = sample(df_all[df_all.source=='v10_patch'], q_v10,  44)
s_base = sample(df_all[df_all.source=='base'],      q_base, 45)

df_train = pd.concat([s_v12, s_v11, s_v10, s_base], ignore_index=True)
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Dataset V12 final : {len(df_train)} exemples")
print(f"   v12_patch : {q_v12} ({q_v12/TARGET*100:.1f}%)")
print(f"   v11_patch : {q_v11} ({q_v11/TARGET*100:.1f}%)")
print(f"   v10_patch : {q_v10} ({q_v10/TARGET*100:.1f}%)")
print(f"   base      : {q_base} ({q_base/TARGET*100:.1f}%)")

# Export
df_export = df_train[["instruction","output"]]
df_export.to_csv("oracle_nlp_dataset.csv", index=False)
df_export.to_csv("oracle_nlp_dataset_v12.csv", index=False)
print(f"\n📄 oracle_nlp_dataset.csv     → {len(df_export)} lignes")
print(f"📄 oracle_nlp_dataset_v12.csv → snapshot sauvegardé")
assert len(df_export) == TARGET, f"ERREUR : {len(df_export)} au lieu de {TARGET}"


Total unique après dédup : 2968
  base      : 860
  v10_patch : 242
  v11_patch : 732
  v12_patch : 1134

✅ Dataset V12 final : 9500 exemples
   v12_patch : 1134 (11.9%)
   v11_patch : 732 (7.7%)
   v10_patch : 242 (2.5%)
   base      : 7392 (77.8%)

📄 oracle_nlp_dataset.csv     → 9500 lignes
📄 oracle_nlp_dataset_v12.csv → snapshot sauvegardé


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 10 — Fine-tuning LoRA V12                           ║
# ║  r=32 alpha=64 · MAX_LEN=256 · 4 époques · lr=1.5e-4        ║
# ║  SYSTEM_PROMPT renforcé avec règles ACTION_NAME explicites   ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch, shutil, os, gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU libre au départ : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

LORA_DIR = "tinyllama_oracle_lora"

# ── SYSTEM_PROMPT V12 — renforcé ──────────────────────────────
# Règles critiques ajoutées vs V11 :
#   - ACTION_NAME='LOGON' TOUJOURS complet (jamais WHERE LOGON seul)
#   - Heures → SYSDATE-N/24, jamais SYSDATE-N pour des heures
#   - Double filtre acteur+objet : WHERE DBUSERNAME=X AND OBJECT_NAME=Y
#   - Poste/machine/terminal → USERHOST
SYSTEM_PROMPT_TRAIN = (
    "Tu es un expert SQL Oracle specialise en audit.\n"
    "Table : ORACLE_AUDIT_TRAIL\n"
    "Colonnes : ID, DBUSERNAME, EVENT_TIMESTAMP, ACTION_NAME, OBJECT_NAME, "
    "USERHOST, RETURNCODE, AUDIT_TYPE, OS_USERNAME.\n"
    "Vue : DBA_USERS (USERNAME, ACCOUNT_STATUS, CREATED).\n"
    "REGLES ABSOLUES :\n"
    "1. Connexions : toujours WHERE ACTION_NAME='LOGON' — jamais WHERE LOGON seul.\n"
    "2. Heures : SYSDATE-N/24. Jours : SYSDATE-N. Minutes : SYSDATE-N/1440.\n"
    "3. Si question sur un utilisateur ET un objet : "
    "WHERE DBUSERNAME='X' AND OBJECT_NAME='Y'.\n"
    "4. Poste/machine/terminal/hote reseau → colonne USERHOST.\n"
    "5. SELECT uniquement. Jamais INSERT/UPDATE/DELETE/DROP en operation reelle.\n"
    "6. Limite : FETCH FIRST N ROWS ONLY.\n"
    "7. Tri par date : ORDER BY EVENT_TIMESTAMP DESC.\n"
    "Reponds uniquement en SQL Oracle valide, sans explication."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Chargement du modèle de base...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.gradient_checkpointing_enable()
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
print("Paramètres entraînables :")
model.print_trainable_parameters()

# ── Dataset ────────────────────────────────────────────────────
raw = load_dataset("csv", data_files="oracle_nlp_dataset.csv")["train"]
print(f"Exemples chargés : {len(raw)}")
assert len(raw) == 9500, f"ERREUR : {len(raw)} exemples au lieu de 9500"

INST_TEMPLATE = (
    f"<|system|>{SYSTEM_PROMPT_TRAIN}<|end|>"
    "<|user|>{instr}<|end|>"
    "<|assistant|>"
)

MAX_LEN = 512

# Vérification longueur prompt
sample = raw[0]
prompt_test = INST_TEMPLATE.format(instr=sample["instruction"])
ids_test = tokenizer(prompt_test)["input_ids"]
print(f"Vérification : prompt_len={len(ids_test)}, MAX_LEN={MAX_LEN}")
assert len(ids_test) < MAX_LEN, f"ERREUR : prompt ({len(ids_test)}) >= MAX_LEN ({MAX_LEN})"
print(f"✅ {MAX_LEN - len(ids_test)} tokens disponibles pour la réponse SQL")

def preprocess(examples):
    full_texts = []
    for instr, out in zip(examples["instruction"], examples["output"]):
        prompt = INST_TEMPLATE.format(instr=instr)
        full_texts.append(prompt + str(out) + "<|end|>")

    tok = tokenizer(
        full_texts,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        add_special_tokens=True,
    )

    labels = []
    for i, (instr, out) in enumerate(zip(examples["instruction"], examples["output"])):
        prompt = INST_TEMPLATE.format(instr=instr)
        prompt_ids = tokenizer(
            prompt, truncation=True, max_length=MAX_LEN, add_special_tokens=True
        )["input_ids"]
        prompt_len = len(prompt_ids)
        label = [-100] * prompt_len + list(tok["input_ids"][i][prompt_len:])
        label = label[:MAX_LEN]
        while len(label) < MAX_LEN:
            label.append(-100)
        labels.append(label)

    tok["labels"] = labels
    return tok

tokenized = raw.map(preprocess, batched=True, remove_columns=["instruction","output"])
tokenized.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

training_args = TrainingArguments(
    output_dir=LORA_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,     # batch effectif = 8
    num_train_epochs=4,
    learning_rate=1.5e-4,
    fp16=True,
    warmup_steps=200,                  # +50 vs V11 pour meilleure convergence
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="epoch",
    report_to=[],
    push_to_hub=False,
    optim="adamw_torch",
    dataloader_num_workers=1,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

print(f"\n🚀 Lancement entraînement V12")
print(f"   Epochs: 4 | LoRA r=32 | MAX_LEN={MAX_LEN} | {len(raw)} exemples")
print(f"   Durée estimée T4 Colab : ~55-65 min")
print(f"   GPU utilisé avant train : {torch.cuda.memory_allocated()/1e9:.2f} GB")

trainer.train()

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"\n✅ Adapter LoRA V12 sauvegardé : {LORA_DIR}/")
shutil.make_archive(LORA_DIR, "zip", LORA_DIR)
print(f"📦 Archive : {LORA_DIR}.zip")


GPU libre au départ : 10.74 GB
Chargement du modèle de base...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Paramètres entraînables :
trainable params: 25,231,360 || all params: 1,125,279,744 || trainable%: 2.2422
Exemples chargés : 9500
Vérification : prompt_len=330, MAX_LEN=512
✅ 182 tokens disponibles pour la réponse SQL


Map:   0%|          | 0/9500 [00:00<?, ? examples/s]


🚀 Lancement entraînement V12
   Epochs: 4 | LoRA r=32 | MAX_LEN=512 | 9500 exemples
   Durée estimée T4 Colab : ~55-65 min
   GPU utilisé avant train : 4.50 GB


Step,Training Loss
50,4.614546
100,0.030365
150,0.010719
200,0.006400
250,0.004561
300,0.002381
350,0.001853
400,0.001145
450,0.001213
500,0.001471


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 11 — Rebuild ZIP LoRA (si kernel redémarré)         ║
# ╚══════════════════════════════════════════════════════════════╝
import os, shutil

LORA_DIR = "tinyllama_oracle_lora"
LORA_ZIP = LORA_DIR + ".zip"

if not os.path.isdir(LORA_DIR):
    raise FileNotFoundError(f"Dossier absent : {LORA_DIR}. Exécuter d'abord la cellule 10.")

required = ["adapter_config.json", "adapter_model.safetensors"]
missing  = [f for f in required if not os.path.exists(os.path.join(LORA_DIR, f))]
if missing:
    raise FileNotFoundError(f"Fichiers LoRA manquants : {missing}")

if os.path.exists(LORA_ZIP):
    os.remove(LORA_ZIP)

shutil.make_archive(LORA_DIR, "zip", LORA_DIR)
size_mb = os.path.getsize(LORA_ZIP) / 1e6
print(f"✅ Archive reconstruite : {LORA_ZIP} ({size_mb:.1f} MB)")
print("   → Télécharger via le panneau Files de Colab")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 12 — Chargement LoRA V12 + fonctions d'inférence    ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, re

LORA_DIR = "tinyllama_oracle_lora"

# SYSTEM_PROMPT identique à l'entraînement (CRITIQUE)
SYSTEM_PROMPT = (
    "Tu es un expert SQL Oracle specialise en audit.\n"
    "Table : ORACLE_AUDIT_TRAIL\n"
    "Colonnes : ID, DBUSERNAME, EVENT_TIMESTAMP, ACTION_NAME, OBJECT_NAME, "
    "USERHOST, RETURNCODE, AUDIT_TYPE, OS_USERNAME.\n"
    "Vue : DBA_USERS (USERNAME, ACCOUNT_STATUS, CREATED).\n"
    "REGLES ABSOLUES :\n"
    "1. Connexions : toujours WHERE ACTION_NAME='LOGON' — jamais WHERE LOGON seul.\n"
    "2. Heures : SYSDATE-N/24. Jours : SYSDATE-N. Minutes : SYSDATE-N/1440.\n"
    "3. Si question sur un utilisateur ET un objet : "
    "WHERE DBUSERNAME='X' AND OBJECT_NAME='Y'.\n"
    "4. Poste/machine/terminal/hote reseau → colonne USERHOST.\n"
    "5. SELECT uniquement. Jamais INSERT/UPDATE/DELETE/DROP en operation reelle.\n"
    "6. Limite : FETCH FIRST N ROWS ONLY.\n"
    "7. Tri par date : ORDER BY EVENT_TIMESTAMP DESC.\n"
    "Reponds uniquement en SQL Oracle valide, sans explication."
)

print(f"Chargement modèle de base : {MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map="auto"
)
print(f"Chargement adapter LoRA : {LORA_DIR}")
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()
print("✅ Modèle V12 chargé.")

INST_TEMPLATE = (
    f"<|system|>{SYSTEM_PROMPT}<|end|>"
    "<|user|>{instr}<|end|>"
    "<|assistant|>"
)

def generate_sql(question: str, max_new_tokens: int = 120) -> str:
    prompt = INST_TEMPLATE.format(instr=question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    sql = gen.split("<|end|>")[0].split("<|user|>")[0].strip()
    return post_process_sql(sql, question)

def post_process_sql(sql: str, question: str) -> str:
    # Fix unicode
    sql = sql.replace("≥",">=").replace("≤","<=")
    q = question.lower()
    su = sql.upper()
    # Fix alias colonnes
    sql = re.sub(r"\bOBJ_NAME\b",   "OBJECT_NAME", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bAUDIT_ID\b",   "ID",          sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bOS_HOST\b",    "USERHOST",    sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bDB_USER\b",    "DBUSERNAME",  sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bTIMESTAMP\b",  "EVENT_TIMESTAMP", sql, flags=re.IGNORECASE)
    # Fix faille α résiduelle : WHERE LOGON → WHERE ACTION_NAME='LOGON'
    sql = re.sub(r"\bWHERE\s+LOGON\b",
                 "WHERE ACTION_NAME='LOGON'", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bAND\s+LOGON\b",
                 "AND ACTION_NAME='LOGON'",   sql, flags=re.IGNORECASE)
    # Fix faille β résiduelle : SYSDATE-48 (sans /24) quand question contient "heure"
    if any(kw in q for kw in ["heure","heures","hres","h ","h,"]):
        def fix_hours(m):
            n = int(m.group(1))
            # Si n est grand (>72), probablement des heures mal converties
            if n > 48 and "jour" not in q:
                return f"SYSDATE-{n}/24"
            return m.group(0)
        sql = re.sub(r"SYSDATE-(\d+)(?!/)", fix_hours, sql, flags=re.IGNORECASE)
    # Remplacement table générique → table réelle
    ORACLE_TABLE = "SMART2DSECU.UNIFIED_AUDIT_DATA"
    sql = re.sub(r"\bORACLE_AUDIT_TRAIL\b", ORACLE_TABLE, sql, flags=re.IGNORECASE)
    return sql

print("✅ Fonctions generate_sql() et post_process_sql() définies.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 13 — Aperçu de la table ORACLE_AUDIT_TRAIL          ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd

AUDIT_DF = pd.read_csv("oracle_audit_trail.csv")

print("╔══════════════════════════════════════════════════════════╗")
print("║         ORACLE_AUDIT_TRAIL — RÉFÉRENCE                   ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Lignes : {len(AUDIT_DF):<5}  Colonnes : {len(AUDIT_DF.columns):<5}                ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  COLONNES :                                               ║")
for col in AUDIT_DF.columns:
    sample_val = str(AUDIT_DF[col].dropna().iloc[0])[:30] if not AUDIT_DF[col].dropna().empty else "NULL"
    print(f"║  {col:<25} ex: {sample_val:<28}║")
print("╚══════════════════════════════════════════════════════════╝")

print(f"\nUtilisateurs distincts : {sorted(AUDIT_DF['DBUSERNAME'].unique())}")
print(f"Actions distinctes     : {sorted(AUDIT_DF['ACTION_NAME'].unique())}")
print(f"Hosts distincts        : {sorted(AUDIT_DF['USERHOST'].dropna().unique())}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 14 — Fusion TinyLlama + LoRA V12                   ║
# ║  Produit un modèle HuggingFace complet avant conversion GGUF ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, shutil, os, gc

MERGED_DIR = "tinyllama_oracle_v12_merged"
LORA_DIR   = "tinyllama_oracle_lora"

gc.collect()
torch.cuda.empty_cache()

if os.path.exists(MERGED_DIR):
    print(f"✅ Modèle fusionné déjà présent : {MERGED_DIR}/")
else:
    print("📥 Chargement modèle de base...")
    tokenizer  = AutoTokenizer.from_pretrained(MODEL_DIR)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        torch_dtype=torch.float16,
        device_map="cpu",   # CPU obligatoire pour merge_and_unload()
    )
    print("🔗 Fusion LoRA...")
    model = PeftModel.from_pretrained(base_model, LORA_DIR)
    model = model.merge_and_unload()
    print(f"💾 Sauvegarde → {MERGED_DIR}/")
    model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    del model, base_model
    gc.collect()
    print(f"✅ Fusion terminée : {MERGED_DIR}/")

files = os.listdir(MERGED_DIR)
size_gb = sum(os.path.getsize(os.path.join(MERGED_DIR, f)) for f in files) / 1e9
print(f"   Fichiers : {files}")
print(f"   Taille totale : {size_gb:.2f} GB")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 15 — Installation llama.cpp                         ║
# ║  Clone du repo + build des outils de conversion              ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, os, sys

LLAMA_DIR = "llama.cpp"

if not os.path.exists(LLAMA_DIR):
    print("📥 Clone llama.cpp...")
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/ggerganov/llama.cpp",
                    LLAMA_DIR], check=True)
    print("✅ llama.cpp cloné.")
else:
    print("✅ llama.cpp déjà présent.")

# Dépendances Python du convertisseur
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentencepiece", "gguf", "transformers", "numpy"], check=True)

# Vérifier le script de conversion
convert_script = None
for root, dirs, files in os.walk(LLAMA_DIR):
    if "convert_hf_to_gguf.py" in files:
        convert_script = os.path.join(root, "convert_hf_to_gguf.py")
        break
print(f"✅ Script de conversion : {convert_script}")

# Build llama-quantize
print("🔨 Build llama-quantize...")
result = subprocess.run(["make", "llama-quantize", "-C", LLAMA_DIR],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("✅ llama-quantize compilé.")
else:
    result2 = subprocess.run(["make", "-C", LLAMA_DIR, "-j4"],
                              capture_output=True, text=True)
    if result2.returncode == 0:
        print("✅ llama.cpp compilé (build complet).")
    else:
        print("⚠️  Build échoué — fallback llama-cpp-python.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "llama-cpp-python"], check=True)

print("✅ Prêt pour la conversion GGUF.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 16 — Conversion HuggingFace → GGUF fp16            ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, os, sys

MERGED_DIR = "tinyllama_oracle_v12_merged"
GGUF_FP16  = "tinyllama_oracle_v12_f16.gguf"
LLAMA_DIR  = "llama.cpp"

convert_script = None
for root, dirs, files in os.walk(LLAMA_DIR):
    if "convert_hf_to_gguf.py" in files:
        convert_script = os.path.join(root, "convert_hf_to_gguf.py")
        break

if convert_script is None:
    raise FileNotFoundError("convert_hf_to_gguf.py introuvable. Exécuter cellule 15.")

if os.path.exists(GGUF_FP16):
    size_gb = os.path.getsize(GGUF_FP16) / 1e9
    print(f"✅ GGUF fp16 déjà présent : {GGUF_FP16} ({size_gb:.2f} GB)")
else:
    print("🔄 Conversion HuggingFace → GGUF fp16...")
    result = subprocess.run([
        sys.executable, convert_script,
        MERGED_DIR,
        "--outfile", GGUF_FP16,
        "--outtype", "f16",
    ], capture_output=True, text=True)

    if result.returncode == 0:
        size_gb = os.path.getsize(GGUF_FP16) / 1e9
        print(f"✅ Conversion réussie : {GGUF_FP16} ({size_gb:.2f} GB)")
    else:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError("Conversion GGUF échouée.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 18 — Benchmark V12 : 10 questions terrain           ║
# ║  Reprend exactement le benchmark utilisé pour scorer V11     ║
# ╚══════════════════════════════════════════════════════════════╝
import re, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ── Rechargement modèle (nécessaire après cellules 14-17) ──────
LORA_DIR = "tinyllama_oracle_lora"

SYSTEM_PROMPT = (
    "Tu es un expert SQL Oracle specialise en audit.\n"
    "Table : ORACLE_AUDIT_TRAIL\n"
    "Colonnes : ID, DBUSERNAME, EVENT_TIMESTAMP, ACTION_NAME, OBJECT_NAME, "
    "USERHOST, RETURNCODE, AUDIT_TYPE, OS_USERNAME.\n"
    "Vue : DBA_USERS (USERNAME, ACCOUNT_STATUS, CREATED).\n"
    "REGLES ABSOLUES :\n"
    "1. Connexions : toujours WHERE ACTION_NAME='LOGON' — jamais WHERE LOGON seul.\n"
    "2. Heures : SYSDATE-N/24. Jours : SYSDATE-N. Minutes : SYSDATE-N/1440.\n"
    "3. Si question sur un utilisateur ET un objet : "
    "WHERE DBUSERNAME='X' AND OBJECT_NAME='Y'.\n"
    "4. Poste/machine/terminal/hote reseau → colonne USERHOST.\n"
    "5. SELECT uniquement. Jamais INSERT/UPDATE/DELETE/DROP en operation reelle.\n"
    "6. Limite : FETCH FIRST N ROWS ONLY.\n"
    "7. Tri par date : ORDER BY EVENT_TIMESTAMP DESC.\n"
    "Reponds uniquement en SQL Oracle valide, sans explication."
)

print("📥 Rechargement du modèle pour le benchmark...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()
print("✅ Modèle V12 rechargé pour benchmark.")

INST_TEMPLATE = (
    f"<|system|>{SYSTEM_PROMPT}<|end|>"
    "<|user|>{instr}<|end|>"
    "<|assistant|>"
)

def generate_sql(question: str, max_new_tokens: int = 120) -> str:
    prompt = INST_TEMPLATE.format(instr=question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    sql = gen.split("<|end|>")[0].split("<|user|>")[0].strip()
    return post_process_sql(sql, question)

def post_process_sql(sql: str, question: str) -> str:
    sql = sql.replace("\u2265",">=").replace("\u2264","<=")
    q = question.lower()
    sql = re.sub(r"\bOBJ_NAME\b",   "OBJECT_NAME", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bAUDIT_ID\b",   "ID",          sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bOS_HOST\b",    "USERHOST",    sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bDB_USER\b",    "DBUSERNAME",  sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bTIMESTAMP\b",  "EVENT_TIMESTAMP", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bWHERE\s+LOGON\b",
                 "WHERE ACTION_NAME='LOGON'", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bAND\s+LOGON\b",
                 "AND ACTION_NAME='LOGON'",   sql, flags=re.IGNORECASE)
    if any(kw in q for kw in ["heure","heures","hres","h ","h,"]):
        def fix_hours(m):
            n = int(m.group(1))
            if n > 48 and "jour" not in q:
                return f"SYSDATE-{n}/24"
            return m.group(0)
        sql = re.sub(r"SYSDATE-(\d+)(?!/)", fix_hours, sql, flags=re.IGNORECASE)
    ORACLE_TABLE = "SMART2DSECU.UNIFIED_AUDIT_DATA"
    sql = re.sub(r"\bORACLE_AUDIT_TRAIL\b", ORACLE_TABLE, sql, flags=re.IGNORECASE)
    return sql

# ── Benchmark ──────────────────────────────────────────────────
AUDIT_DF = pd.read_csv("oracle_audit_trail.csv").fillna("")

BENCHMARK = [
    {
        "id": "Q1",
        "question": "Qui s'est connecté hier ?",
        "must_contain": ["ACTION_NAME='LOGON'", "SYSDATE-1"],
        "must_not_contain": ["WHERE LOGON"],
        "description": "Faille α : ACTION_NAME='LOGON' explicite + SYSDATE-1"
    },
    {
        "id": "Q2",
        "question": "Quels utilisateurs se sont connectés dans les 48 dernières heures ?",
        "must_contain": ["ACTION_NAME='LOGON'", "/24"],
        "must_not_contain": ["SYSDATE-48 "],
        "description": "Faille β : 48h → SYSDATE-48/24"
    },
    {
        "id": "Q3",
        "question": "Qu'est-ce qui s'est passé en janvier 2026 ?",
        "must_contain": ["2026-01-01", "TO_DATE"],
        "must_not_contain": ["TRUNC(SYSDATE,'MM')"],
        "description": "Mois nommé → TO_DATE bornes explicites"
    },
    {
        "id": "Q4",
        "question": "Quelle table a été la plus modifiée ?",
        "must_contain": ["OBJECT_NAME", "GROUP BY", "INSERT", "UPDATE", "DELETE"],
        "must_not_contain": ["GROUP BY DBUSERNAME"],
        "description": "GROUP BY OBJECT_NAME (pas DBUSERNAME)"
    },
    {
        "id": "Q5",
        "question": "Qui a accédé à la table PAYROLL ?",
        "must_contain": ["OBJECT_NAME='PAYROLL'", "DBUSERNAME"],
        "must_not_contain": ["ACTION_NAME='LOGON'"],
        "description": "Accès objet → SELECT/DML, pas LOGON"
    },
    {
        "id": "Q6",
        "question": "Quelles actions VROMUALD a-t-il effectuées sur la table EMPLOYEES ?",
        "must_contain": ["DBUSERNAME='VROMUALD'", "OBJECT_NAME='EMPLOYEES'"],
        "must_not_contain": [],
        "description": "Faille δ : double filtre acteur + objet"
    },
    {
        "id": "Q7",
        "question": "Quel poste de travail a effectué le plus de connexions ?",
        "must_contain": ["USERHOST", "ACTION_NAME='LOGON'", "GROUP BY"],
        "must_not_contain": ["GROUP BY DBUSERNAME"],
        "description": "Faille ε : poste → USERHOST"
    },
    {
        "id": "Q8",
        "question": "Y a-t-il eu des suppressions de comptes utilisateurs ?",
        "must_contain": ["ACTION_NAME='DROP USER'"],
        "must_not_contain": [],
        "description": "Langage métier → DROP USER"
    },
    {
        "id": "Q9",
        "question": "Qui a le plus changé d'informations au cours des 30 derniers jours ?",
        "must_contain": ["INSERT","UPDATE","DELETE","SYSDATE-30","GROUP BY"],
        "must_not_contain": [],
        "description": "Langage métier + fenêtre temporelle"
    },
    {
        "id": "Q10",
        "question": "Quels utilisateurs se sont connectés entre 22h et 6h du matin ?",
        "must_contain": ["ACTION_NAME='LOGON'","HH24","22","6"],
        "must_not_contain": [],
        "description": "Horaire nocturne → TO_CHAR HH24"
    },
]

def evaluate(sql: str, must_contain: list, must_not_contain: list) -> dict:
    su = sql.upper()
    hits   = [kw for kw in must_contain     if kw.upper() in su]
    misses = [kw for kw in must_contain     if kw.upper() not in su]
    bads   = [kw for kw in must_not_contain if kw.upper() in su]
    score  = len(hits) / len(must_contain) * 100 if must_contain else 100
    if bads:
        score = max(0, score - 30 * len(bads))
    return {"score": round(score,1), "hits": hits, "misses": misses, "bads": bads}

print("="*72)
print("  BENCHMARK V12 — 10 QUESTIONS TERRAIN")
print("="*72)

results = []
total_score = 0

for test in BENCHMARK:
    sql = generate_sql(test["question"])
    res = evaluate(sql, test["must_contain"], test["must_not_contain"])

    status = "✅" if res["score"] >= 80 else ("⚠️ " if res["score"] >= 50 else "❌")
    print(f"\n{status} {test['id']} — {test['description']}")
    print(f"   Question : {test['question']}")
    print(f"   SQL      : {sql[:120]}{'...' if len(sql)>120 else ''}")
    print(f"   Score    : {res['score']}%")
    if res["misses"]: print(f"   Manque   : {res['misses']}")
    if res["bads"]:   print(f"   Indésir. : {res['bads']}")

    total_score += res["score"]
    results.append({
        "id": test["id"],
        "description": test["description"],
        "score": res["score"],
        "sql": sql,
        "misses": res["misses"],
        "bads": res["bads"],
    })

avg = total_score / len(BENCHMARK)
print("\n" + "="*72)
print(f"  SCORE GLOBAL V12 : {avg:.1f}%")
target_ok = "✅ OBJECTIF 90% ATTEINT" if avg >= 90 else f"⚠️  Objectif 90% — encore {90-avg:.1f}% à gagner"
print(f"  {target_ok}")
print("="*72)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 18 — Benchmark V12 : 10 questions terrain           ║
# ║  Reprend exactement le benchmark utilisé pour scorer V11     ║
# ╚══════════════════════════════════════════════════════════════╝
import re, pandas as pd

AUDIT_DF = pd.read_csv("oracle_audit_trail.csv").fillna("")
HH24_SIM = "TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24'))"

# 10 questions du benchmark terrain — correspondance exacte V11
BENCHMARK = [
    {
        "id": "Q1",
        "question": "Qui s'est connecté hier ?",
        "must_contain": ["ACTION_NAME='LOGON'", "SYSDATE-1"],
        "must_not_contain": ["WHERE LOGON"],
        "description": "Faille α : ACTION_NAME='LOGON' explicite + SYSDATE-1"
    },
    {
        "id": "Q2",
        "question": "Quels utilisateurs se sont connectés dans les 48 dernières heures ?",
        "must_contain": ["ACTION_NAME='LOGON'", "/24"],
        "must_not_contain": ["SYSDATE-48 "],   # sans /24
        "description": "Faille β : 48h → SYSDATE-48/24"
    },
    {
        "id": "Q3",
        "question": "Qu'est-ce qui s'est passé en janvier 2026 ?",
        "must_contain": ["2026-01-01", "TO_DATE"],
        "must_not_contain": ["TRUNC(SYSDATE,'MM')"],
        "description": "Mois nommé → TO_DATE bornes explicites"
    },
    {
        "id": "Q4",
        "question": "Quelle table a été la plus modifiée ?",
        "must_contain": ["OBJECT_NAME", "GROUP BY", "INSERT", "UPDATE", "DELETE"],
        "must_not_contain": ["GROUP BY DBUSERNAME"],
        "description": "GROUP BY OBJECT_NAME (pas DBUSERNAME)"
    },
    {
        "id": "Q5",
        "question": "Qui a accédé à la table PAYROLL ?",
        "must_contain": ["OBJECT_NAME='PAYROLL'", "DBUSERNAME"],
        "must_not_contain": ["ACTION_NAME='LOGON'"],
        "description": "Accès objet → SELECT/DML, pas LOGON"
    },
    {
        "id": "Q6",
        "question": "Quelles actions VROMUALD a-t-il effectuées sur la table EMPLOYEES ?",
        "must_contain": ["DBUSERNAME='VROMUALD'", "OBJECT_NAME='EMPLOYEES'"],
        "must_not_contain": [],
        "description": "Faille δ : double filtre acteur + objet"
    },
    {
        "id": "Q7",
        "question": "Quel poste de travail a effectué le plus de connexions ?",
        "must_contain": ["USERHOST", "ACTION_NAME='LOGON'", "GROUP BY"],
        "must_not_contain": ["GROUP BY DBUSERNAME"],
        "description": "Faille ε : poste → USERHOST"
    },
    {
        "id": "Q8",
        "question": "Y a-t-il eu des suppressions de comptes utilisateurs ?",
        "must_contain": ["ACTION_NAME='DROP USER'"],
        "must_not_contain": [],
        "description": "Langage métier → DROP USER"
    },
    {
        "id": "Q9",
        "question": "Qui a le plus changé d'informations au cours des 30 derniers jours ?",
        "must_contain": ["INSERT","UPDATE","DELETE","SYSDATE-30","GROUP BY"],
        "must_not_contain": [],
        "description": "Langage métier + fenêtre temporelle"
    },
    {
        "id": "Q10",
        "question": "Quels utilisateurs se sont connectés entre 22h et 6h du matin ?",
        "must_contain": ["ACTION_NAME='LOGON'","HH24","22","6"],
        "must_not_contain": [],
        "description": "Horaire nocturne → TO_CHAR HH24"
    },
]

def evaluate(sql: str, must_contain: list, must_not_contain: list) -> dict:
    su = sql.upper()
    hits   = [kw for kw in must_contain     if kw.upper() in su]
    misses = [kw for kw in must_contain     if kw.upper() not in su]
    bads   = [kw for kw in must_not_contain if kw.upper() in su]
    score  = len(hits) / len(must_contain) * 100 if must_contain else 100
    if bads:
        score = max(0, score - 30 * len(bads))  # pénalité -30% par must_not trouvé
    return {"score": round(score,1), "hits": hits, "misses": misses, "bads": bads}

print("="*72)
print("  BENCHMARK V12 — 10 QUESTIONS TERRAIN")
print("="*72)

results = []
total_score = 0

for test in BENCHMARK:
    sql = generate_sql(test["question"])
    res = evaluate(sql, test["must_contain"], test["must_not_contain"])

    status = "✅" if res["score"] >= 80 else ("⚠️ " if res["score"] >= 50 else "❌")
    print(f"\n{status} {test['id']} — {test['description']}")
    print(f"   Question : {test['question']}")
    print(f"   SQL      : {sql[:120]}{'...' if len(sql)>120 else ''}")
    print(f"   Score    : {res['score']}%")
    if res["misses"]: print(f"   Manque   : {res['misses']}")
    if res["bads"]:   print(f"   Indésir. : {res['bads']}")

    total_score += res["score"]
    results.append({
        "id": test["id"],
        "description": test["description"],
        "score": res["score"],
        "sql": sql,
        "misses": res["misses"],
        "bads": res["bads"],
    })

avg = total_score / len(BENCHMARK)
print("\n" + "="*72)
print(f"  SCORE GLOBAL V12 : {avg:.1f}%")
target_ok = "✅ OBJECTIF 90% ATTEINT" if avg >= 90 else f"⚠️  Objectif 90% — encore {90-avg:.1f}% à gagner"
print(f"  {target_ok}")
print("="*72)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 19 — Rapport de performance global                  ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
from datetime import datetime

try:
    df_results = pd.DataFrame(results)
    avg_score = df_results["score"].mean()

    print("╔══════════════════════════════════════════════════════════╗")
    print("║          RAPPORT DE PERFORMANCE — MODÈLE V12             ║")
    print(f"║          Généré le {datetime.now().strftime('%Y-%m-%d %H:%M')}                       ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print(f"║  Score global     : {avg_score:.1f}%                              ║")
    print(f"║  Questions testées: {len(df_results)}                                   ║")
    print(f"║  Réussies (≥80%)  : {len(df_results[df_results.score>=80])}                                   ║")
    print(f"║  Partielles (50-79%): {len(df_results[(df_results.score>=50)&(df_results.score<80)])}                                 ║")
    print(f"║  Échouées (<50%)  : {len(df_results[df_results.score<50])}                                   ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print("║  ÉVOLUTION :                                              ║")
    print("║  V9  → 30%   V10 → (non testé)                           ║")
    print("║  V11 → 50%   V12 → {:.1f}%                              ║".format(avg_score))
    print("╚══════════════════════════════════════════════════════════╝")

    print("\nDétail par question :")
    for _, row in df_results.iterrows():
        status = "✅" if row["score"] >= 80 else ("⚠️ " if row["score"] >= 50 else "❌")
        print(f"  {status} {row['id']:4s} {row['score']:5.1f}%  {row['description']}")
        if row["misses"]: print(f"         Manque: {row['misses']}")

    df_results.to_csv("rapport_performance_v12.csv", index=False)
    print("\n📄 rapport_performance_v12.csv sauvegardé.")

    if avg_score < 90:
        print("\n" + "─"*60)
        print("  ANALYSE FAILLES RESTANTES — Pistes V13")
        print("─"*60)
        for _, row in df_results[df_results.score < 80].iterrows():
            print(f"  {row['id']} : score={row['score']}% | manque={row['misses']}")
        print("  → Générer Bloc M avec des exemples ciblés sur ces failles.")
except NameError:
    print("⚠️  Exécuter d'abord la cellule 14 (benchmark).")


## Déploiement Local — Fichiers à récupérer depuis Colab

| Fichier | Description |
|---|---|
| `TinyLlama-1.1B-Chat-v1.0.zip` | Modèle de base (~2.2 GB) |
| `tinyllama_oracle_lora.zip` | Adapter LoRA **V12** (r=32, MAX_LEN=256) |
| `oracle_audit_trail.csv` | Table simulée (500 lignes) |
| `oracle_nlp_dataset_v12.csv` | Dataset V12 (9 500 exemples) |
| `rapport_performance_v12.csv` | Résultats benchmark |

### Commandes pip (environnement local)

```bash
python -m venv venv_asksmart
source venv_asksmart/bin/activate  # Linux/Mac
# ou venv_asksmart\Scripts\activate  # Windows

pip install transformers>=4.40.0 peft>=0.10.0 accelerate>=0.28.0
pip install torch>=2.2.0 pandas>=2.0.0 python-oracledb
```

### Remplacer la table dans l'app FastAPI

Dans le backend ASKSMART, `post_process_sql()` remplace automatiquement `ORACLE_AUDIT_TRAIL` par `SMART2DSECU.UNIFIED_AUDIT_DATA` — ne pas modifier ce mécanisme.

### SYSTEM_PROMPT dans app_queryflow / ASKSMART backend

Le SYSTEM_PROMPT dans le backend doit être **identique** à celui utilisé à l'entraînement (cellule 10). Copier-coller la constante `SYSTEM_PROMPT` de la cellule 12.
